<a href="https://colab.research.google.com/github/a-nagar/cs4372/blob/main/Recommendation_Frequent_Pattern/Recommender_Systems_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Recommender Systems with Latent Factor Models
### Computational Methods for Data Science (CS 4372)

This assignment builds on the recommendation lab. You will use the **`surprise`** and **`scikit-learn`** libraries (no algorithms to implement yourself) to build, tune, and use recommender systems on three well-known datasets:

| Part | Dataset | Topic | Points |
|---|---|---|---|
| 1 | MovieLens 100K | Exploring the data; collaborative filtering (KNN) | 20 |
| 2 | MovieLens 100K | **Latent factor models**: SVD, SVD++, NMF; number of factors; grid search | 30 |
| 3 | MovieLens 100K | **Using the latent factors**: recommendations and similar items (`surprise` and `scikit-learn`) | 15 |
| 4 | MovieLens 1M | Scaling up to 1 million ratings | 15 |
| 5 | Goodbooks-10k | A new domain: book ratings from Goodreads | 15 |
| 6 | All | Comparing results across datasets | 5 |
| | | **Total** | **100** |

**Instructions**
* Complete every cell marked `# TODO`. Most TODOs are only a few lines: you call a library function and display its results.
* Keep the random seeds (`random_state=42` / `random_state=0`) so everyone's numbers are comparable.
* Answer the written questions (**Q*x.y***) in the markdown cells, referring to your own outputs.
* The notebook should run top to bottom in Colab in about 10 minutes. Submit the **executed** `.ipynb`.

**Two ideas to keep in mind**
* **Collaborative filtering (neighborhood methods)** predict a rating from the ratings of similar users or similar items.
* **Latent factor models** learn a short vector $p_u$ for each user and $q_i$ for each item, and predict $\hat r_{ui} = \mu + b_u + b_i + p_u^\top q_i$ (global mean + user bias + item bias + taste match). `surprise.SVD` fits this model with stochastic gradient descent.

## Setup

In [ ]:
!pip install -q scikit-surprise

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors

from surprise import Dataset, Reader, get_dataset_dir, accuracy
from surprise import NormalPredictor, BaselineOnly, KNNBasic, KNNWithMeans, KNNBaseline, SVD, SVDpp, NMF
from surprise.model_selection import train_test_split, cross_validate, GridSearchCV

pd.set_option("display.max_colwidth", 60)
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]),
    "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2,
})

### Helper functions (provided — read them, but you do not need to change them)

In [ ]:
def evaluate(algo, trainset, testset, name=None):
    # Fit algo on trainset, predict testset, and return RMSE, MAE and fit time
    t0 = time.time()
    algo.fit(trainset)
    fit_time = time.time() - t0
    preds = algo.test(testset)
    return {"model": name or type(algo).__name__,
            "RMSE": accuracy.rmse(preds, verbose=False),
            "MAE": accuracy.mae(preds, verbose=False),
            "fit time (s)": round(fit_time, 2)}

def get_top_n(algo, trainset, user_raw_id, n=10):
    # Predict every item the user has NOT rated in the training set; return the n highest predictions
    inner_u = trainset.to_inner_uid(user_raw_id)
    seen = {j for (j, _) in trainset.ur[inner_u]}
    preds = [(trainset.to_raw_iid(j), algo.predict(user_raw_id, trainset.to_raw_iid(j)).est)
             for j in trainset.all_items() if j not in seen]
    return sorted(preds, key=lambda x: -x[1])[:n]

def similar_items(algo, trainset, item_raw_id, n=10, min_ratings=0):
    # Items whose latent vectors q_i are closest (cosine similarity) to the given item (needs an SVD-type model)
    Q = algo.qi / np.linalg.norm(algo.qi, axis=1, keepdims=True)
    i = trainset.to_inner_iid(item_raw_id)
    sims = Q @ Q[i]
    counts = np.array([len(trainset.ir[j]) for j in trainset.all_items()])
    order = [j for j in np.argsort(-sims) if j != i and counts[j] >= min_ratings][:n]
    return [(trainset.to_raw_iid(j), round(float(sims[j]), 3)) for j in order]

---
## Part 1 — MovieLens 100K: Data and Collaborative Filtering (20 pts)

MovieLens 100K has 100,000 ratings (1–5 stars) from 943 users on 1,682 movies. It is built into `surprise`.

In [ ]:
data = Dataset.load_builtin("ml-100k", prompt=False)
ratings = pd.DataFrame(data.raw_ratings, columns=["user", "item", "rating", "timestamp"])

# movie titles (raw item ids in surprise are strings)
movies = pd.read_csv(get_dataset_dir() + "/ml-100k/ml-100k/u.item", sep="|", encoding="ISO-8859-1",
                     header=None, usecols=[0, 1], names=["item", "title"], dtype={"item": str})
id_to_title = dict(zip(movies.item, movies.title))
title_to_id = dict(zip(movies.title, movies.item))
ratings.head()

### 1.1 Explore the data (5 pts)
Report the number of users, items, and ratings, and the **density** (ratings ÷ (users × items)). Plot a histogram of the ratings, and show the 10 most-rated movies with their titles and average ratings.

In [ ]:
# TODO: number of users, items, ratings, and the density

# TODO: histogram (or bar chart) of the rating values

# TODO: the 10 most-rated movies with title, count and mean rating
#       hint: ratings.groupby("item").rating.agg(["count", "mean"]) and id_to_title

**Q1.1** About what fraction of the user–movie matrix is filled in? What does that mean for a recommender system?

**Your answer:**

_Write your answer here._

### 1.2 Train/test split and baselines (5 pts)
The split below is provided; use it for all of Parts 1–3. Then evaluate two simple baselines with `evaluate`:
* `NormalPredictor()`: predicts a random rating drawn from the overall rating distribution
* `BaselineOnly()`: predicts $\mu + b_u + b_i$ (global mean + user bias + item bias, no personalization beyond that)

Collect every result dictionary in the list `results_100k`.

In [ ]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
results_100k = []

In [ ]:
# TODO: evaluate NormalPredictor() and BaselineOnly(verbose=False); append each result to results_100k

### 1.3 Collaborative filtering with KNN (7 pts)
Evaluate **`KNNBasic`** and **`KNNWithMeans`** (both with `k=40`) for every combination of
* `user_based`: `True` (user–user CF) or `False` (item–item CF)
* similarity `name`: `"cosine"` or `"pearson"`

That is 8 models. Show them in a table sorted by RMSE, and add the best one to `results_100k`.

In [ ]:
knn_rows = []
for Algo in [KNNBasic, KNNWithMeans]:
    for user_based in [True, False]:
        for sim in ["cosine", "pearson"]:
            name = f"{Algo.__name__} | {'user' if user_based else 'item'}-based | {sim}"
            # TODO: create the algorithm with k=40 and the right sim_options, evaluate it, and append to knn_rows
            ...
# TODO: knn_table sorted by RMSE; append the best row (as a dict) to results_100k

**Q1.3** (a) Does subtracting each user's mean (`KNNWithMeans` vs `KNNBasic`) help? Why would it? (b) Which works better here, user-based or item-based? Did any configuration beat `BaselineOnly`?

**Your answer:**

_Write your answer here._

### 1.4 Item neighbors (3 pts)
As in the lab, fit `KNNBaseline` with `sim_options={"name": "pearson_baseline", "user_based": False}` on `trainset`, and print the 10 nearest neighbors of **Star Wars (1977)** using `get_neighbors`. Convert ids with `trainset.to_inner_iid`, `trainset.to_raw_iid`, and `id_to_title`.

In [ ]:
# TODO: fit KNNBaseline (item-based, pearson_baseline) on trainset; also evaluate it and append to results_100k
# TODO: print the titles of the 10 nearest neighbors of "Star Wars (1977)"

**Q1.4** Do these neighbors make sense? Compare them with the neighbors you got in the lab on the 1,000-rating Book-Crossing sample, and explain the difference.

**Your answer:**

_Write your answer here._

---
## Part 2 — Latent Factor Models on MovieLens 100K (30 pts)

### 2.1 SVD, SVD++, and NMF (8 pts)
Evaluate these three latent factor models with default parameters (`random_state=0`) and add them to `results_100k`:
* `SVD`: the biased matrix factorization model $\hat r_{ui} = \mu + b_u + b_i + p_u^\top q_i$
* `SVDpp`: SVD++, which also uses *which* items a user rated (implicit feedback). Pass `cache_ratings=True` for speed.
* `NMF`: non-negative matrix factorization (all factors ≥ 0, no bias terms)

Then print the shapes of the fitted SVD model's `pu`, `qi`, `bu`, and `bi` attributes.

In [ ]:
# TODO: evaluate SVD, SVDpp and NMF; append to results_100k and display the three new rows
# TODO: print the shapes of svd.pu, svd.qi, svd.bu, svd.bi

**Q2.1** (a) Explain the shapes of `pu`, `qi`, `bu`, and `bi`. What is the default number of latent factors? (b) How many parameters does the SVD model learn in total, and how does that compare with the number of training ratings? (c) Which of the three models is most accurate, and which is slowest to train?

**Your answer:**

_Write your answer here._

### 2.2 How many latent factors? (8 pts)
Train `SVD(n_factors=k, random_state=0)` for $k \in \{1, 2, 5, 10, 20, 50, 100, 200\}$. For each $k$, record the **training** RMSE (predict on `trainset.build_testset()`) and the **test** RMSE. Plot both curves against $k$ (log x-axis).

In [ ]:
train_eval = trainset.build_testset()
factor_rows = []
for k in [1, 2, 5, 10, 20, 50, 100, 200]:
    # TODO: fit SVD(n_factors=k, random_state=0); record train RMSE (on train_eval) and test RMSE
    ...
# TODO: DataFrame indexed by k, and a plot of both curves (log x-axis)

**Q2.2** Describe the two curves. What happens to training error as $k$ grows? What happens to test error? Which $k$ would you choose, and why is the gap between the curves a sign of overfitting?

**Your answer:**

_Write your answer here._

### 2.3 Tune SVD with `GridSearchCV` (10 pts)
Using **only the training data**, run `GridSearchCV` with `cv=3`, `measures=["rmse", "mae"]`, and `n_jobs=-1` on the grid below. Report the best parameters and best CV RMSE, then refit an SVD with those parameters on `trainset`, evaluate it on `testset`, and add it to `results_100k` as `"SVD (tuned)"`.

`GridSearchCV` needs a `Dataset`, so the cell below rebuilds one from the training ratings only (so the test set is never used for tuning).

In [ ]:
train_df = pd.DataFrame([(trainset.to_raw_uid(u), trainset.to_raw_iid(i), r) for (u, i, r) in trainset.all_ratings()],
                        columns=["user", "item", "rating"])
train_data = Dataset.load_from_df(train_df, Reader(rating_scale=(1, 5)))

param_grid = {"n_factors": [20, 50, 100], "n_epochs": [20, 40], "lr_all": [0.005, 0.01],
              "reg_all": [0.02, 0.1], "random_state": [0]}

In [ ]:
# TODO: GridSearchCV(SVD, param_grid, measures=["rmse", "mae"], cv=3, n_jobs=-1) on train_data
# TODO: print best score and best params (save the params as best_params; you will reuse them in Parts 4 and 5)
# TODO: svd_best = SVD(**best_params); evaluate on trainset/testset; append to results_100k as "SVD (tuned)"

**Q2.3** (a) Which hyperparameters changed from the defaults (`n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02`), and how much did the test RMSE improve? (b) Why must tuning use only the training data and not the test set?

**Your answer:**

_Write your answer here._

### 2.4 Leaderboard (4 pts)
Show all of `results_100k` as a table sorted by RMSE, and as a horizontal bar chart of RMSE (provided).

In [ ]:
board_100k = pd.DataFrame(results_100k).drop_duplicates("model").set_index("model").sort_values("RMSE")
display(board_100k.round(4))

fig, ax = plt.subplots(figsize=(8, 0.4 * len(board_100k) + 1))
ax.barh(board_100k.index[::-1], board_100k.RMSE[::-1], height=0.6, color="#2a78d6")
for y, v in enumerate(board_100k.RMSE[::-1]):
    ax.text(v + 0.005, y, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlim(0.85, board_100k.RMSE.max() + 0.08); ax.grid(axis="y", visible=False)
ax.set_xlabel("test RMSE (lower is better)"); ax.set_title("MovieLens 100K"); plt.show()

**Q2.4** Rank the families of models (random, baseline, KNN, latent factor). Roughly how much of the improvement over random guessing comes from the biases alone (`BaselineOnly`), and how much more do the best latent factor models add?

**Your answer:**

_Write your answer here._

---
## Part 3 — Using the Latent Factors (15 pts)

### 3.1 Top-10 recommendations (5 pts)
Use the tuned model `svd_best` and the helper `get_top_n` to produce the top-10 movie recommendations for **user `"196"`**. Print the titles and predicted ratings, and, for context, the user's 10 highest-rated movies in `train_df`.

In [ ]:
user = "196"
# TODO: print the user's 10 highest-rated movies in train_df (titles and ratings)
# TODO: print the top-10 recommendations from get_top_n(svd_best, trainset, user, n=10) with titles

**Q3.1** Do the recommendations look personalized, or are they mostly movies everyone likes? Which term in $\hat r_{ui} = \mu + b_u + b_i + p_u^\top q_i$ pushes widely acclaimed movies to the top for *every* user?

**Your answer:**

_Write your answer here._

### 3.2 Similar movies from latent factors (`surprise`) (4 pts)
Use the helper `similar_items(svd_best, trainset, item_id, n=8, min_ratings=50)` to find the movies most similar to **Star Wars (1977)**, **Toy Story (1995)**, and **Scream (1996)**. Print titles and similarity scores. (`min_ratings=50` skips rarely-rated movies.)

In [ ]:
for t in ["Star Wars (1977)", "Toy Story (1995)", "Scream (1996)"]:
    # TODO: print the 8 most similar movies (with similarity) using similar_items(..., min_ratings=50)
    ...

### 3.3 Latent factors with `scikit-learn` (6 pts)
You can also get item factors by factoring the rating matrix directly with `sklearn.decomposition.TruncatedSVD` (classical SVD on a sparse matrix, treating missing ratings as 0).

1. The code below builds the sparse **item × user** matrix `M` (rows = movies), with each movie's ratings centered by that movie's mean.
2. Fit `TruncatedSVD(n_components=20, random_state=42)` and transform `M` to get a 20-dimensional vector per movie.
3. Fit `NearestNeighbors(metric="cosine")` on those vectors and print the 8 nearest neighbors of the same three movies, restricted to movies with ≥ 50 ratings.
4. Print the fraction of variance explained by the 20 components (`explained_variance_ratio_.sum()`).

In [ ]:
r = train_df.copy()
r["centered"] = r.rating - r.groupby("item").rating.transform("mean")
item_ids = np.sort(r["item"].unique())
user_ids = np.sort(r["user"].unique())
row = pd.Index(item_ids).get_indexer(r["item"])
col = pd.Index(user_ids).get_indexer(r["user"])
M = csr_matrix((r["centered"].values, (row, col)), shape=(len(item_ids), len(user_ids)))
counts = r.groupby("item").size().reindex(item_ids).values
print("item x user matrix:", M.shape, " stored entries:", M.nnz)

In [ ]:
# TODO: fit TruncatedSVD(n_components=20, random_state=42) on M -> item_vecs; print the explained variance

popular = np.flatnonzero(counts >= 50)
# TODO: NearestNeighbors(metric="cosine") fitted on item_vecs[popular]
for t in ["Star Wars (1977)", "Toy Story (1995)", "Scream (1996)"]:
    q = item_vecs[[np.flatnonzero(item_ids == title_to_id[t])[0]]]
    # TODO: nn.kneighbors(q, n_neighbors=9); skip the first neighbor (the movie itself); print 1 - distance and title
    ...

**Q3.3** (a) Compare the neighbor lists from 3.2 (`surprise` SVD) and 3.3 (`scikit-learn` TruncatedSVD). Which seem more sensible? (b) The two methods differ in how they treat *missing* ratings. Explain the difference, and why it matters for a matrix that is ~94% empty.

**Your answer:**

_Write your answer here._

---
## Part 4 — Scaling Up: MovieLens 1M (15 pts)

MovieLens 1M has about 1 million ratings from 6,040 users on about 3,700 movies. Load it from the GitHub mirror below (tab-separated).

In [ ]:
ML1M = "https://raw.githubusercontent.com/khanhnamle1994/movielens/master/"
ml1m = pd.read_csv(ML1M + "ratings.csv", sep="\t", usecols=["user_id", "movie_id", "rating"])
ml1m_movies = pd.read_csv(ML1M + "movies.csv", sep="\t", usecols=["movie_id", "title", "genres"], encoding="ISO-8859-1")
ml1m_title = dict(zip(ml1m_movies.movie_id, ml1m_movies.title))
ml1m.head()

### 4.1 Explore and compare with 100K (3 pts)
Report users, items, ratings, density, and the average number of ratings per user and per item, side by side with MovieLens 100K, as a small table.

In [ ]:
# TODO: a table comparing users, items, ratings, density, ratings/user and ratings/item for the two datasets

### 4.2 Models (8 pts)
Build a `surprise` dataset from `ml1m`, split it 80/20 with `random_state=42`, and evaluate (collect the results in `results_1m`):
* `BaselineOnly`
* `SVD` with default parameters
* `SVD(**best_params)` (the parameters you tuned on MovieLens 100K)
* `SVD(n_factors=50, n_epochs=30, lr_all=0.005, reg_all=0.05, random_state=0)`
* `KNNBaseline` item-based with `pearson_baseline` (takes about a minute)

In [ ]:
data_1m = Dataset.load_from_df(ml1m[["user_id", "movie_id", "rating"]], Reader(rating_scale=(1, 5)))
trainset_1m, testset_1m = train_test_split(data_1m, test_size=0.2, random_state=42)

# TODO: evaluate the five models into results_1m, and show board_1m sorted by RMSE

### 4.3 Recommendations (1 pt)
Using the best SVD model from 4.2 (refit it if needed), print the top-10 recommendations for user `1` with titles.

In [ ]:
# TODO: fit your best SVD configuration on trainset_1m and print user 1's top-10 recommendations with titles

**Q4** (a) Compare the RMSEs with MovieLens 100K. Why are all the models more accurate on MovieLens 1M? (b) Did the hyperparameters tuned on 100K transfer well to 1M? What does that tell you about tuning? (c) Item-based KNN is competitive here. What are its practical drawbacks compared with SVD as the catalog grows (think about fit time and memory)?

**Your answer:**

_Write your answer here._

---
## Part 5 — A New Domain: Goodbooks-10k (15 pts)

[Goodbooks-10k](https://github.com/zygmuntz/goodbooks-10k) contains about 6 million ratings (1–5 stars) from about 53,000 Goodreads users on the 10,000 most popular books. To keep things fast, we use a random sample of **8,000 users** (provided).

In [ ]:
GB = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/"
gb_all = pd.read_csv(GB + "ratings.csv")
gb_books = pd.read_csv(GB + "books.csv", usecols=["book_id", "title", "authors", "average_rating"])
gb_title = dict(zip(gb_books.book_id, gb_books.title))
gb_id = dict(zip(gb_books.title, gb_books.book_id))

rng = np.random.default_rng(42)
sample_users = rng.choice(gb_all.user_id.unique(), size=8000, replace=False)
gb = gb_all[gb_all.user_id.isin(sample_users)].reset_index(drop=True)
print(f"full data: {len(gb_all):,} ratings   sample: {len(gb):,} ratings")
gb.head()

### 5.1 Explore (3 pts)
Add Goodbooks to the comparison table from 4.1 (use your `summarize` function on `gb`), and plot the rating distributions of MovieLens 1M and Goodbooks side by side as proportions.

In [ ]:
# TODO: comparison table with all three datasets
# TODO: side-by-side bar chart of the rating distributions (as proportions) for MovieLens 1M and Goodbooks

### 5.2 Models (6 pts)
Split `gb` 80/20 (`random_state=42`) and evaluate, collecting results in `results_gb`:
* `BaselineOnly`
* `SVD` with default parameters
* `SVD(**best_params)`
* `SVD(n_factors=150, n_epochs=40, lr_all=0.007, reg_all=0.08, random_state=0)`

(Item-based KNN would need a 10,000 × 10,000 similarity matrix, so we skip it here.)

In [ ]:
data_gb = Dataset.load_from_df(gb[["user_id", "book_id", "rating"]], Reader(rating_scale=(1, 5)))
trainset_gb, testset_gb = train_test_split(data_gb, test_size=0.2, random_state=42)

# TODO: evaluate the four models into results_gb (keep the last SVD as svd_gb for 5.3), and show board_gb

### 5.3 Similar books and recommendations (3 pts)
Using `svd_gb` and the helpers:
1. Print the 8 books most similar to **"Harry Potter and the Sorcerer's Stone (Harry Potter, #1)"** and **"The Hunger Games (The Hunger Games, #1)"** (use `min_ratings=100`).
2. Pick the user with the most ratings in `gb`, and print their five highest-rated books and their top-10 recommendations.

In [ ]:
for t in ["Harry Potter and the Sorcerer's Stone (Harry Potter, #1)", "The Hunger Games (The Hunger Games, #1)"]:
    # TODO: 8 most similar books using similar_items(svd_gb, trainset_gb, gb_id[t], n=8, min_ratings=100)
    ...

# TODO: the user with the most ratings in gb: their 5 highest-rated books and their top-10 recommendations

**Q5** (a) How do the Goodbooks rating distribution and density differ from MovieLens? (b) How much does SVD improve over `BaselineOnly` on Goodbooks compared with MovieLens 1M? Give one reason why personalization adds less here. (c) Do the similar-book lists make sense? What kinds of relationships do the latent factors seem to capture (series, author, genre, audience)? (d) Look at the top-10 recommendations for the heaviest reader. Do they make sense? Why do niche titles appear with predicted ratings near 5?

**Your answer:**

_Write your answer here._

---
## Part 6 — Comparing Across Datasets (5 pts)
Build one table with, for each dataset: the `BaselineOnly` RMSE, the best model's name and RMSE, and the **improvement of the best model over `BaselineOnly`** in percent. Then answer Q6.

In [ ]:
# TODO: one row per dataset: BaselineOnly RMSE, best model and RMSE, and % improvement over the baseline

**Q6** In 4–6 sentences: across the three datasets, how much does a latent factor model add on top of a simple bias baseline, and what properties of a dataset seem to make personalization more or less valuable? If you were building a book recommender for a new website with very few ratings, which model would you start with, and why?

**Your answer:**

_Write your answer here._

---
### References
* Koren, Bell & Volinsky (2009). *Matrix Factorization Techniques for Recommender Systems*. IEEE Computer.
* Harper & Konstan (2015). *The MovieLens Datasets: History and Context*. ACM TiiS.
* Zajac (2017). *Goodbooks-10k: a new dataset for book recommendations*. https://github.com/zygmuntz/goodbooks-10k
* `surprise` documentation: https://surprise.readthedocs.io · `scikit-learn` TruncatedSVD: https://scikit-learn.org/stable/modules/decomposition.html#lsa